# 문제 13. Martian Map: 화성 탐사 관심 지형 탐지

**난이도 최상 목표 · 제한시간 없음 · CPU 전용 · Python3 · CSV 제출**

## 상황

화성 탐사팀은 궤도영상을 분석해 향후 과학 관측 후보를 검토할 지도를 준비하고 있습니다. 영상에서 **원뿔형 지형(cone)**을 찾아 사각형 박스로 표시하세요. 여기서 cone은 중앙에 구덩이가 있는 원뿔형 지형이며 도로용 안전콘이나 로버 앞의 작은 돌이 아닙니다.

이 문제는 실제 화성 궤도영상과 연구진의 지형 주석을 활용한 학습용 과제입니다. 탐사 지도에 추가할 영상 내 관심 지형 목록을 만드는 단계까지 다룹니다. 위도·경도 지도 통합, 경로 최적화, 보행 안전 판정, 현재 기상 분석은 과제에 포함하지 않습니다. 2차원 외관만으로 지형의 생성 원인이나 생명체 존재를 확정하지 않습니다.

## Data

모든 영상은 **512×512 흑백 PNG**입니다. 영상 한 장에 목표가 없을 수도 있고 여러 개 있을 수도 있습니다. 원본 영상을 잘라 얻은 패치이므로 패치 수와 독립 촬영 횟수는 다릅니다.

| 구분 | 영상 수 | 객체가 있는 영상 | 배경 영상 | 정답 박스 수 |
|---|---:|---:|---:|---:|
| fit | 7,059 | 903 | 6,156 | 2,041 |
| valid | 2,209 | 454 | 1,755 | 859 |

테스트 영상은 **4,418장**입니다. 테스트의 객체 수와 객체 유무는 제공하지 않습니다.

| 파일 | 내용 |
|---|---|
| data/train.csv | image_id, image_path, scene_id, width, height, split |
| data/train_boxes.csv | 학습·검증의 객체별 image_id, class_id, x_min, y_min, x_max, y_max |
| data/test.csv | 테스트 이미지 목록. split·정답 없음 |
| data/images/ | 전체 이미지 |
| data/labels/ | 학습·검증 이미지의 YOLO 정답 TXT. 배경은 빈 파일, 테스트 라벨은 없음 |
| data/sample_submission.csv | 제출 컬럼만 있는 빈 CSV. 탐지 결과에 따라 행 수가 달라짐 |
| models/yolov8n_coco.pt | 선택적으로 사용할 수 있는 일반 이미지 사전학습 초기 가중치 |
| yolo_helpers.py | 제공 분할을 YOLO 학습 목록과 YAML로 연결하는 준비 함수 |
| evaluation.py | 검증용 채점 함수와 제출 형식 검사. 숨겨진 정답 없음 |
| training_examples.jpg | fit 이미지와 정답 박스 예시 |

### 클래스와 좌표

클래스는 하나이며 **class_id=0, cone**입니다. 사각형은 원본 제공 영상의 픽셀 좌표로 표현합니다. 왼쪽 위가 (0,0), 오른쪽 아래 경계가 (512,512)입니다.

- `0 ≤ x_min < x_max ≤ 512`, `0 ≤ y_min < y_max ≤ 512`.
- 너비는 `x_max-x_min`, 높이는 `y_max-y_min`입니다. +1을 더하지 않습니다.
- 영상 경계에서 잘린 지형도 주석이 있으면 평가 대상입니다. 작은 주석을 임의로 삭제하지 마세요.
- YOLO TXT의 한 행은 `0 x_center y_center width height`이며 좌표·크기는 영상 크기로 나눈 0~1 값입니다. CSV와 단위가 다릅니다.
- `train_boxes.csv`에 등장하지 않는 학습 이미지는 정답 객체가 없는 배경입니다. 데이터 누락이 아닙니다.

### 학습·검증 분리

`scene_id`는 원본 지도 타일 그룹입니다. fit·valid·test는 서로 다른 그룹이며, 이미지 ID와 scene_id는 예측 특징으로 쓰지 마세요. 제공된 split을 사용하고 이미지 패치를 무작위로 다시 나누지 마세요. 다른 그룹 사이의 모든 지리적 중첩까지 완전히 제거했다는 의미는 아닙니다.

검증 중에는 fit만 모델 학습에 사용합니다. 전처리·학습량·해상도·신뢰도·중복 제거 설정은 valid로 선택합니다. 최종 설정을 정한 뒤에는 fit+valid로 재학습할 수 있지만, 그때 valid 점수는 독립 검증 성능이 아닙니다.

## 초기 코드

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image

BASE_DIR = Path('.').resolve()
train = pd.read_csv(BASE_DIR / 'data/train.csv')
train_boxes = pd.read_csv(BASE_DIR / 'data/train_boxes.csv')
test = pd.read_csv(BASE_DIR / 'data/test.csv')
sample_submission = pd.read_csv(BASE_DIR / 'data/sample_submission.csv')
display(train.head())
display(train_boxes.head())
display(Image.open(BASE_DIR / train.iloc[0]['image_path']))

## 과제

1. 학습 이미지와 정답을 함께 확인하고, 배경 비율·박스 크기·잘린 객체·그룹별 영상 차이를 분석하세요.
2. YOLO 등 객체 탐지 모델을 **CPU에서** 학습하거나 미세 조정하세요. 작은 모델, 입력 해상도, 배치 크기, 동결할 층과 학습 자료 구성을 판단하세요.
3. valid에서 위치 정확도와 오탐·미탐을 확인하고 성능을 개선하세요. 학습 중 배경을 일부 추출할 수 있지만 검증은 전체 valid로 평가해야 합니다.
4. test 전체를 예측해 `submission.csv`를 저장하세요. 검출이 없는 이미지에는 행을 쓰지 않습니다.

### CPU 및 모델 사용

- 제한시간은 없습니다. GPU·CUDA·MPS 및 원격 GPU 학습은 사용하지 않습니다. 학습·추론에 `device='cpu'`를 명시하세요.
- Windows/Jupyter에서는 `workers=0`으로 시작하세요. 작은 모델과 작은 배치로 동작을 확인하고 필요하면 확장하세요. CPU 종류에 따라 학습 시간은 크게 달라집니다.
- 제공 모델은 일반 사진으로 사전학습한 모델이며 **이 문제의 cone을 이미 학습한 모델이 아닙니다.** 기존 COCO 클래스 번호를 0으로 바꾸는 것은 cone 학습을 대신하지 않습니다.
- 다른 모델과 사전학습 가중치, 추가 라이브러리, 미세 조정을 허용합니다. 다만 테스트 영상·정답을 학습에 사용하거나 원본 공개 정답을 찾아 제출하지 마세요. 외부 모델이 평가 영상을 학습했는지도 확인하세요.
- 학습과 추론 코드는 Notebook에 포함하고 실행 결과를 저장하세요. 별도 보고서는 요구하지 않습니다.

### 제공 준비 함수 — 선택 사항

`from yolo_helpers import prepare_yolo`

`prepare_yolo(BASE_DIR, BASE_DIR / 'yolo_work')`는 fit/valid 목록과 `dataset.yaml`을 만들고 YAML 경로를 반환합니다. 데이터 다운로드·학습·예측은 수행하지 않습니다. 기본값은 모든 fit 이미지입니다.

`negative_ratio=1.0`을 지정하면 fit의 모든 양성 이미지와 그 수만큼의 배경을 고정 시드로 추출합니다. valid는 줄이지 않습니다. 배경을 줄이는 것이 반드시 성능 향상을 보장하지는 않습니다.

### 평가

**주 점수 = 100 × AP50–95**입니다. 하나의 cone 클래스에 대해 IoU 0.50, 0.55, …, 0.95의 10개 기준에서 AP를 구하고 평균합니다. AP50·AP75는 보조 지표입니다.

- IoU는 예측 박스와 정답 박스의 교집합 면적 / 합집합 면적입니다.
- 각 IoU 기준에서 신뢰도가 높은 예측부터 같은 이미지의 아직 매칭되지 않은 정답 중 IoU가 가장 큰 하나에 대응시킵니다. 기준 미달·중복 탐지는 오탐입니다.
- 전체 영상의 예측을 신뢰도 순서로 모아 정밀도–재현율 곡선을 계산합니다. 0~1의 재현율 101개 지점에서 보간 정밀도를 평균합니다.
- 모든 박스 크기를 평가하며 별도의 crowd·ignore 영역은 없습니다. 배경 영상의 예측도 오탐으로 평가합니다.
- 이미지당 **최대 100개** 예측을 제출할 수 있습니다. 낮은 신뢰도 예측을 많이 삭제하면 최대 재현율이 낮아질 수 있습니다.
- 신뢰도 동률은 이미지 목록 순서, 같은 이미지 안에서는 CSV 행 순서로 처리합니다. 정답 IoU 동률은 제공 정답 행 순서를 따릅니다.
- 검증은 `evaluation.evaluate(predictions, train_boxes, train[train.split == 'valid'])`로 계산하세요. 형식과 정의가 다른 라이브러리 점수보다 이 함수가 과제의 기준입니다.

이 과제는 작은/잘린 객체, 배경 불균형, 타일 간 분포 차이와 정밀한 위치 예측을 함께 다루는 최상 목표 과제입니다. 낮은 점수나 긴 학습 시간 자체를 최상 난도의 근거로 삼지 않으며, 공식 대회 난도·합격선과의 동등성을 보장하지 않습니다.

## 풀이

In [ ]:
# 여기에 코드를 작성하세요.

## 결과 저장

Notebook과 같은 폴더에 **submission.csv**를 저장하세요.

컬럼 순서는 반드시 `image_id,class_id,confidence,x_min,y_min,x_max,y_max`입니다.

| image_id | class_id | confidence | x_min | y_min | x_max | y_max |
|---|---:|---:|---:|---:|---:|---:|
| test.csv에 있는 이미지 ID | 0 | 0.82 | 120.5 | 80.0 | 205.0 | 160.0 |

위 행은 형식 예시이며 정답이 아닙니다. 신뢰도는 0~1의 유한한 수, 좌표는 원본 512×512 영상 기준입니다. 정규화 좌표·타일 내부 좌표를 그대로 제출하지 마세요. `index=False`로 저장하세요.

검출이 없는 이미지는 행을 생략하며 `class_id=-1` 같은 더미 행을 넣지 않습니다. 전체가 빈 헤더뿐인 제출도 형식은 유효하지만 점수는 0입니다. 알 수 없는 ID, 범위를 벗어난 좌표, 결측값, 완전히 동일한 중복 행은 오류입니다.

제출 행 수는 고정되어 있지 않습니다. 제출 CSV의 순서는 자유지만, 동점 처리에 영향을 줄 수 있습니다. 아래 검사는 저장 형식만 확인하며 점수를 계산하거나 외부에 제출하지 않습니다.

In [ ]:
# 여기에 모델 학습·검증·추론 코드를 작성하세요.
# predictions.to_csv(BASE_DIR / 'submission.csv', index=False)

## 제출 형식 검사

In [ ]:
from evaluation import validate

submission_path = BASE_DIR / 'submission.csv'
if submission_path.exists():
    submitted = pd.read_csv(submission_path)
    validate(submitted, test)
    print(f'형식 검사 통과: {len(submitted):,}개 탐지. 점수 검사는 아닙니다.')
else:
    print('아직 submission.csv가 없습니다. 풀이 후 저장하세요.')